<a href="https://colab.research.google.com/github/rsher60/LLM_Codebase/blob/main/fine_tune_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# This notebook is to push a training dataset to hugging face.

Prompt & Schema reference: https://huggingface.co/datasets/davanstrien/ufo-ColPali

In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

In [ ]:
!pip install pydantic

## I will be using the OpenAI GPT to add the questions to the image . The goal of this would be generate a dataset on HuggingFace and then use that dataset to fine tune ColPali.

In [ ]:
prompt_2 = """You are an AI assistant specialized in document retrieval tasks. Given an image of a document page, your task is to generate retrieval queries that someone might use to find this document in a large corpus.

Please generate 3 different types of retrieval queries:

1. A broad topical query: This should cover the main subject of the document.
2. A specific detail query: This should focus on a particular fact, figure, or point made in the document.
3. A visual element query: This should reference a chart, graph, image, or other visual component in the document, if present.

Important guidelines:
- Ensure the queries are relevant for retrieval tasks, not just describing the page content.
- Frame the queries as if someone is searching for this document, not asking questions about its content.
- Make the queries diverse and representative of different search strategies.

For each query, also provide a brief explanation of why this query would be effective in retrieving this document.

Format your response as a JSON object with the following structure:

{
  "broad_topical_query": "Your query here",
  "broad_topical_explanation": "Brief explanation",
  "specific_detail_query": "Your query here",
  "specific_detail_explanation": "Brief explanation",
  "visual_element_query": "Your query here",
  "visual_element_explanation": "Brief explanation"
}

If there are no relevant visual elements, replace the third query with another specific detail query.

Here is the document image to analyze:
<image>

Generate the queries based on this image and provide the response in the specified JSON format."""

In [ ]:
#!pip install langchain-openai pillow langchain-core

In [ ]:
import base64
import os
from PIL import Image
import pathlib
from langchain_openai import ChatOpenAI
import io
from langchain_core.messages import HumanMessage, SystemMessage

def convert_png_to_bytes(png_path):
    """
    Reads a PNG image file from the given path and returns its content as bytes.
    """
    try:
        with open(png_path, 'rb') as f:
            png_bytes = f.read()
        return png_bytes
    except FileNotFoundError:
        print(f"Error: PNG file not found at {png_path}")
        return None
    except Exception as e:
        print(f"An error occurred while reading the PNG file: {e}")
        return None

#Helper function to encode the images
def encode_image(image_bytes):
    # Open the image using PIL
    image = Image.open(io.BytesIO(image_bytes))

    # Re-encode the image as JPEG (or other format)
    buffered = io.BytesIO()
    image.save(buffered, format="JPEG")  # Or PNG, GIF etc.  JPEG is a good default.

    # Base64 encode the re-encoded image
    img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")

    return img_str

#Helper function to use GPT 4o LLM
def summarize_image(img_base64 , prompt):
    """ This function will leverage GPT 4 to summarize the base64 encoded images
    """
    chat = ChatOpenAI(model="gpt-4o",max_tokens = 1024, api_key= OPENAI_API_KEY)
    msg = chat.invoke(
        [
            HumanMessage(
                content=[
                    {"type":"text","text":prompt},
                    {
                        "type":"image_url",
                        "image_url": {"url":f"data:image/jpeg;base64,{img_base64}"}
                    },

                ]
            )
        ]
    )

    return msg.content



#Main function to call GPT and summarize the uploaded image
def get_image_summary(file_name: str) -> str:
    """Generates a concise summary of an image.
    Args:
        file_name: The path to the image file.
    Returns:
        The image summary as a string.
    """
    prompt = """You are an assistant tasked with summarizing images for retrieval.\
        These summaries will be embedded and used to retreive the raw images.\
        Give a consice summary of the image that is well optimised for retrieval and storage in a vector"""

    try:
        # Check if the file is a valid image using pathlib
        image_path = pathlib.Path(file_name)
        if image_path.is_file() and image_path.suffix.lower() in ('.jpg', '.jpeg', '.png'):
            base64_image = encode_image(file_name)
            return summarize_image(base64_image, prompt)
        else:
            raise ValueError("Unsupported file format")
    except (ValueError, OSError) as e:
        return f"Error: {e}"

In [ ]:
import pandas as pd
dataframe = pd.DataFrame()

for i in


a = encode_image(convert_png_to_bytes('/content/drive/My Drive/Colab Notebooks/GEP-Jan-2024_images/GEP-Jan-2024.pdf_page_26.png'))
summarize_image(a
                , prompt_2)




In [ ]:
from PIL import Image
import numpy as np

# Define the path to your PNG file
image_path = '/content/drive/My Drive/Colab Notebooks/GEP-Jan-2024_images/GEP-Jan-2024.pdf_page_26.png' # Replace with the actual path to your PNG

try:
    # Open the image file
    image = Image.open(image_path)

    # (Optional) Convert the image to a NumPy array for numerical processing
    image_array = np.array(image)

    # You can now work with the 'image' object or 'image_array'
    print(f"Image format: {image.format}")
    print(f"Image size: {image.size}")
    print(f"Image mode: {image.mode}")
    if 'image_array' in locals():
        print(f"Image array shape: {image_array.shape}")

    # (Optional) Display the image (requires matplotlib)
    import matplotlib.pyplot as plt
    plt.imshow(image)
    plt.show()

except FileNotFoundError:
    print(f"Error: Image file not found at {image_path}")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
import os

# Define the path to the directory you want to list
# For example, if your files are in a subfolder named 'data' within your Colab environment:
directory_path = '/content/drive/My Drive/Colab Notebooks/GEP-Jan-2024_images'

# Or, if you've mounted Google Drive and want to access a folder there:
# from google.colab import drive
# drive.mount('/content/drive')
# directory_path = '/content/drive/MyDrive/Your_Folder_Name'

# Get a list of all entries (files and subdirectories) in the specified directory
file_list = os.listdir(directory_path)

# Print the list of files
print(len(file_list))

240


In [ ]:
import os

import pandas as pd
dataframe = pd.DataFrame()


filename_col = []
qa_col = []




# Define the directory you want to search
# Replace 'My Drive/Your_Colab_Folder' with the actual path to your folder
folder_path = '/content/drive/My Drive/Colab Notebooks/GEP-Jan-2024_images'

# --- Example 1: Show all files in the folder ---
print(f"All files in '{folder_path}':")

try:

  all_files = os.listdir(folder_path)
  for file_name in all_files:
    if os.path.isfile(os.path.join(folder_path, file_name)): # Ensure it's a file, not a subdirectory
      print(file_name)
      filename_col.append(file_name)
      a = encode_image(convert_png_to_bytes(f'/content/drive/My Drive/Colab Notebooks/GEP-Jan-2024_images/{file_name}'))

      qa_col.append(str(summarize_image(a, prompt_2)))

except FileNotFoundError:


  print(f"Error: Folder '{folder_path}' not found.")

print("\n" + "="*30 + "\n")


if len(filename_col) == len(qa_col):
  print("yes they are equal")
  df = pd.DataFrame([filename_col, qa_col]).T

  df.columns = ['filename_col','qa_col']

  df.head()

All files in '/content/drive/My Drive/Colab Notebooks/GEP-Jan-2024_images':
GEP-Jan-2024.pdf_page_79.png
GEP-Jan-2024.pdf_page_36.png
GEP-Jan-2024.pdf_page_14.png
GEP-Jan-2024.pdf_page_41.png
GEP-Jan-2024.pdf_page_144.png
GEP-Jan-2024.pdf_page_97.png
GEP-Jan-2024.pdf_page_111.png
GEP-Jan-2024 (1).pdf_page_85.png
GEP-Jan-2024 (1).pdf_page_52.png
GEP-Jan-2024.pdf_page_12.png
GEP-Jan-2024.pdf_page_93.png
GEP-Jan-2024.pdf_page_86.png
GEP-Jan-2024.pdf_page_125.png
GEP-Jan-2024.pdf_page_121.png
GEP-Jan-2024.pdf_page_151.png
GEP-Jan-2024.pdf_page_153.png
GEP-Jan-2024.pdf_page_32.png
GEP-Jan-2024.pdf_page_115.png
GEP-Jan-2024 (1).pdf_page_91.png
GEP-Jan-2024.pdf_page_103.png
GEP-Jan-2024.pdf_page_66.png
GEP-Jan-2024.pdf_page_85.png
GEP-Jan-2024.pdf_page_108.png
GEP-Jan-2024.pdf_page_138.png
GEP-Jan-2024.pdf_page_49.png
GEP-Jan-2024.pdf_page_69.png
GEP-Jan-2024.pdf_page_72.png
GEP-Jan-2024.pdf_page_6.png
GEP-Jan-2024 (1).pdf_page_149.png
GEP-Jan-2024.pdf_page_47.png
GEP-Jan-2024.pdf_page_137.pn

In [ ]:
df.to_csv('qa_pairs_gep.csv')

In [ ]:
! pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 16.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; pl

In [ ]:
import pandas as pd

df = pd.read_csv('qa_pairs_gep.csv')

In [ ]:
df.head()
df = df.drop('Unnamed: 0',axis=1)

In [ ]:
import json
def strip_unwanted_elements(text_string):
    """
    Strips out specific characters and the word 'json' from a given string.

    Args:
        text_string (str): The input string to be processed.

    Returns:
        str: The modified string with unwanted elements removed.
    """
    # Define the characters and words to be removed
    to_strip = [
        '```',  # Backticks
        "'''",  # Triple single quotes
        'json', # The word 'json'
        '\n'    # Newline characters
    ]

    # Iterate through the list of elements to strip and replace them
    # with an empty string
    for item in to_strip:
        #if item.count('{')==1 and item.count('}')==1:
        text_string = text_string.replace(item, '')

    #print(text_string.count('{'))

    if text_string.count('{')==1 and text_string.count('}')==1:
      result_dict = json.loads(text_string)

    else:
      result_dict = "{}"

    return result_dict








{'broad_topical_query': 'Global economic prospects January 2024',
 'broad_topical_explanation': 'This query is effective because it directly references the main subject and time frame of the document, which are likely indexed in a corpus for retrieval.',
 'specific_detail_query': 'Growth in advanced economies projected to slow in 2024',
 'specific_detail_explanation': 'This query targets a specific prediction mentioned in the document, which distinguishes this report and makes it identifiable in a large database of economic reports.',
 'visual_element_query': 'Global real interest rate cycles and consumer price inflation charts',
 'visual_element_explanation': "This query is focused on retrieving the document based on the visual elements, specifically the charts labeled 'C. U.S. real interest rate cycles' and 'B. Global consumer price inflation,' making it easier to find this document among others with similar subjects."}

In [ ]:
type(df['qa_col'][1])

str

In [ ]:
df = pd.read_csv('qa_pairs_gep.csv')
df = df.drop('Unnamed: 0',axis=1)
df['qa_col_modified']  = df['qa_col'].apply(lambda x:strip_unwanted_elements(str(x)))

# I am getting this error because the JSON error is taking an empty string.

In [ ]:
import pandas as pd

def extract_qa_details(dicty):
    """
    Extracts values from the dictionary in 'qa_col_modified' for new columns.
    Handles missing dictionaries or keys gracefully.
    """


    # Initialize a dictionary to hold the extracted values
    extracted_values = {
        'broad_topical_query': None,
        'broad_topical_explanation': None,
        'specific_detail_query': None,
        'specific_detail_explanation': None,
        'visual_element_query': None,
        'visual_element_explanation': None
    }

    if isinstance(dicty, dict):
        #print (type(dicty))
        for key in extracted_values.keys():
            extracted_values[key] = dicty.get(key)

    return pd.Series(extracted_values)




extracted_df = df['qa_col_modified'].apply(extract_qa_details)

In [ ]:
extracted_df.head()

# Concatenate the original DataFrame with the newly extracted columns
df_final = pd.concat([df, extracted_df], axis=1)

In [ ]:
df_final.shape

(240, 9)

In [ ]:
x = df_final[df_final['qa_col_modified'] !='{}']
x = x.drop(['qa_col', 'qa_col_modified'], axis=1)
#x.head()
x = x.rename(columns={'filename_col': 'filename'})

x.to_csv('final_colpali_dataset.csv')

In [ ]:
from datasets import load_dataset, Dataset
from google.colab import drive
import os
from PIL import Image
import pandas as pd
import glob
from huggingface_hub import login

login()

# Mount Google Drive
drive.mount('/content/drive')

# Paths
local_data_path = "/content/drive/My Drive/Colab Notebooks/GEP-Jan-2024_images"
csv_path = "final_colpali_dataset.csv"  # Update this path

# Alternative path without spaces (create a symlink)
symlink_path = "/content/images_data_main"

# Create symlink to avoid path issues
if not os.path.exists(symlink_path):
    os.symlink(local_data_path, symlink_path)
    print(f"Created symlink: {symlink_path} -> {local_data_path}")

# Verify paths exist
if not os.path.exists(local_data_path):
    print(f"Image path does not exist: {local_data_path}")
    exit()

if not os.path.exists(csv_path):
    print(f"CSV path does not exist: {csv_path}")
    print("Please update the csv_path variable with the correct path to your CSV file")
    exit()

try:
    # Step 1: Load the CSV with questions and answers
    print("Loading CSV data...")
    qa_df = pd.read_csv(csv_path)
    print(f"CSV loaded with {len(qa_df)} rows")
    print("CSV columns:", qa_df.columns.tolist())
    print("First few rows:")
    print(qa_df.head())

    # Step 2: Load images
    print("\nLoading images...")

    # Get all image files manually
    image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff', '*.webp']
    image_paths = []

    for ext in image_extensions:
        # Check subdirectories
        pattern = os.path.join(local_data_path, '**', ext)
        image_paths.extend(glob.glob(pattern, recursive=True))
        # Also check direct files in the folder
        pattern = os.path.join(local_data_path, ext)
        image_paths.extend(glob.glob(pattern))

    print(f"Found {len(image_paths)} images")

    if len(image_paths) == 0:
        print("No images found! Check your image directory.")
        exit()

    # Step 3: Create image dataset
    def load_image_data(path):
        try:
            filename = os.path.basename(path)
            return {
                "image": Image.open(path),
                "image_path": path,
                "image_filename": filename
            }
        except Exception as e:
            print(f"Error loading {path}: {e}")
            return None

    print("Processing images...")
    image_data = []
    for path in image_paths:
        img_data = load_image_data(path)
        if img_data:
            image_data.append(img_data)

    print(f"Successfully loaded {len(image_data)} images")

    # Step 4: Combine images with CSV data
    print("\nCombining images with CSV data...")

    # Create image DataFrame
    image_df = pd.DataFrame([{
        'image_filename': item['image_filename'],
        'image': item['image'],
        'image_path': item['image_path']
    } for item in image_data])

    # You need to specify how to match images with CSV rows
    # Common approaches:

    # Option 1: If CSV has a column with image filenames
    if 'image_filename' in qa_df.columns or 'filename' in qa_df.columns:
        merge_column = 'image_filename' if 'image_filename' in qa_df.columns else 'filename'
        combined_df = pd.merge(qa_df, image_df, left_on=merge_column, right_on='image_filename', how='inner')
        print(f"Merged data: {len(combined_df)} rows")

    # Option 2: If CSV rows correspond to images by index/order
    elif len(qa_df) == len(image_df):
        print("Assuming CSV rows correspond to images by order...")
        combined_df = pd.concat([qa_df.reset_index(drop=True), image_df.reset_index(drop=True)], axis=1)
        print(f"Combined data: {len(combined_df)} rows")

    # Option 3: Repeat CSV data for each image (if you want each image paired with all Q&A)
    else:
        print("Creating cartesian product of images and Q&A pairs...")
        qa_repeated = []
        images_repeated = []

        for _, qa_row in qa_df.iterrows():
            for img_data in image_data:
                qa_repeated.append(qa_row.to_dict())
                images_repeated.append(img_data)

        # Combine into single list of dictionaries
        combined_data = []
        for qa, img in zip(qa_repeated, images_repeated):
            combined_row = {**qa, **img}  # Merge dictionaries
            combined_data.append(combined_row)

        combined_df = pd.DataFrame(combined_data)
        print(f"Combined data: {len(combined_df)} rows")

    # Step 5: Create Hugging Face Dataset
    print("\nCreating Hugging Face dataset...")

    # Convert DataFrame to list of dictionaries for Dataset
    dataset_data = combined_df.to_dict('records')

    # Create the dataset
    dataset = Dataset.from_list(dataset_data)
    print("Dataset created successfully:")
    print(dataset)

    # Step 6: Upload to Hugging Face
    dataset_name = 'rsher60/colpali-finetuning-dataset-gep2'
    dataset_description = "Sample dataset for ColPali finetuning using GEP report from World Economic Forum - combines images with Q&A pairs"

    try:
        print(f"\nUploading dataset to {dataset_name}...")
        dataset.push_to_hub(
            repo_id=dataset_name,

        )
        print(f"✅ Dataset uploaded successfully!")
        print(f"🔗 Dataset URL: https://huggingface.co/datasets/{dataset_name}")
        print(f"📝 You can now load it with: load_dataset('{dataset_name}')")

    except Exception as upload_error:
        print(f"❌ Failed to upload dataset: {upload_error}")
        print("Make sure you're logged in and have the correct permissions.")

except Exception as e:
    print(f"Error in processing: {e}")
    import traceback
    traceback.print_exc()